In [ ]:
import pandas as pd
s = pd.read_csv("sales_transactions.csv", parse_dates=["date"])
p = pd.read_csv("promotion_calendar.csv", parse_dates=["start_date", "end_date"])
s.info()
p.info()
s.describe()

In [ ]:
s["disc"] = s.discount_rate.where(s.discount_rate <= 1, s.discount_rate / 100)
s["month"] = s.date.dt.month
s["gross"] = s.quantity * s.unit_price
s["net"] = s.gross * (1 - s.disc)
s["profit"] = s.net - s.quantity * s.unit_cost
s["diff"] = s.net_revenue - s.net
s["cost"] = s.gross * s.disc
s["gap15"] = (0.15 * s.net - s.profit).clip(lower=0)
s[s["diff"].abs() > 0.01]

In [ ]:
g = s.groupby(["category", "month"]).disc
q1, q3, n = g.transform("quantile", 0.25), g.transform("quantile", 0.75), g.transform("size")
s["iqr"] = (n >= 20) & (s.disc > q3 + 1.5 * (q3 - q1))
print((n < 20).sum(), (s.disc == 0).mean().round(2))
s[s.iqr]

In [ ]:
v = s[s.promotion_id.notna()].merge(p, how="left")
v["id_ok"] = v.max_discount.notna()
v["cat_ok"] = v.category == v.applicable_category
v["date_ok"] = v.date.between(v.start_date, v.end_date)
v["max_ok"] = v.disc <= v.max_discount
v["plan_ok"] = (v.disc - v.planned_discount).abs().round(4) <= 0.05
c = ["id_ok", "cat_ok", "date_ok", "max_ok", "plan_ok"]
bad = v[v[c].all(axis=1) == False]
bad[["promotion_id"] + c]

In [ ]:
ok = v[v.id_ok & v.cat_ok & v.date_ok].transaction_id
s["valid"] = s.transaction_id.isin(ok)
s["promo_issue"] = s.transaction_id.isin(bad.transaction_id)
s["over30"] = (s.disc > 0.3) & (s.valid == False)
s["low"] = s.gap15 > 0
s[s.over30]

In [ ]:
def impact(by):
    t = s.groupby(by)[["net", "profit", "cost", "gap15"]].sum()
    t["n"] = s.groupby(by).size()
    t["margin"] = t.profit / t.net
    return t.round(2)
cat = impact(["category", "valid"])
w = cat.xs(True, level=1).net
print(round((cat.xs(False, level=1).margin * w).sum() / w.sum(), 3))
display(impact("valid"), cat)
pr = impact(s.promotion_id.where(s.valid))
pr

In [ ]:
plan = s.promotion_id.map(p.set_index("promotion_id").planned_discount)
s["excess"] = (s.gross * (s.disc - plan.where(s.valid, g.transform("median")))).clip(lower=0)
u = s.iqr | s.over30 | s.promo_issue
money = pd.Series({"cost": s.cost.sum(), "unusual": s[u].excess.sum(), "gap15": s.gap15.sum(), "prm07": pr.gap15["PRM-07"]}).round(2)
money

In [ ]:
k = ["net_revenue", "net", "diff"]
r = s.groupby("category")[k].sum()
r.loc["Sum"] = r.sum()
r.loc["Total"] = s[k].sum()
r.round(2)

In [ ]:
s["data_issue"] = (s["diff"].abs() > 0.01) | (s.discount_rate > 1)
f = ["iqr", "data_issue", "promo_issue", "over30", "low"]
s[s[f].any(axis=1)].to_csv("flagged_transactions.csv", index=False)
s.assign(discount_raw=s.discount_rate, discount_rate=s.disc).drop(columns="disc").to_csv("sales_transactions.csv", index=False)
p.merge(pr.reset_index(), how="left").to_csv("promotion_calendar.csv", index=False)
res = pd.concat([s[f].sum(), money])
res.to_csv("summary_results.csv")
res

## Summary
- Discounts cost $35,095; unusual ones only $1,073 above normal.
- PRM-07 explains $3,449 of the $3,870 below the 15% floor.
- Promotion margin: 17% vs 31% at the same category mix.
- Inputs fixed and updated. Files generated with AI.